In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/interim/cms_suppliers_cleaned.csv")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
df.dtypes

Rows: 57197
Columns: ['provider_id', 'acceptsassignement', 'participationbegindate', 'practicecity', 'practicestate', 'specialitieslist', 'supplieslist', 'latitude', 'longitude', 'acceptsassignement_bool', 'AcceptsNumeric', 'participationbegindate_parsed', 'date_reliability_flag']


provider_id                        int64
acceptsassignement                  bool
participationbegindate            object
practicecity                      object
practicestate                     object
specialitieslist                  object
supplieslist                      object
latitude                         float64
longitude                        float64
acceptsassignement_bool             bool
AcceptsNumeric                     int64
participationbegindate_parsed     object
date_reliability_flag             object
dtype: object

In [2]:
feature_roles = pd.DataFrame([
    {"column": "provider_id", "role": "Identifier", "used_as_feature": False, "reason": "Unique ID - explicitly excluded per rule: do not use provider ID as a predictive feature"},
    {"column": "acceptsassignement", "role": "Target (raw)", "used_as_feature": False, "reason": "Original target string - use acceptsassignement_bool instead"},
    {"column": "acceptsassignement_bool", "role": "Target", "used_as_feature": False, "reason": "This is what we are predicting"},
    {"column": "AcceptsNumeric", "role": "Target (numeric)", "used_as_feature": False, "reason": "Duplicate encoding of target - not a feature"},
    {"column": "participationbegindate", "role": "Date", "used_as_feature": False, "reason": "Raw string date - use derived features instead"},
    {"column": "participationbegindate_parsed", "role": "Date", "used_as_feature": False, "reason": "Datetime object itself is not directly usable - will derive tenure_days feature instead"},
    {"column": "date_reliability_flag", "role": "Derived/Quality flag", "used_as_feature": "Under review", "reason": "5,469 rows have suspected placeholder dates; including this risks the model just learning our own data-quality flag rather than a real pattern - decide after checking correlation"},
    {"column": "practicecity", "role": "Geography", "used_as_feature": False, "reason": "Too high-cardinality (thousands of distinct cities) for reliable encoding without overfitting; practicestate captures geography at a usable grain"},
    {"column": "practicestate", "role": "Geography", "used_as_feature": True, "reason": "55 distinct values - reasonable for one-hot or target encoding"},
    {"column": "specialitieslist", "role": "Specialty", "used_as_feature": True, "reason": "Multi-value categorical - will be encoded via multi-hot/binary indicator columns per specialty"},
    {"column": "supplieslist", "role": "Supply category", "used_as_feature": True, "reason": "Multi-value categorical - same encoding approach as specialty"},
    {"column": "latitude", "role": "Geography", "used_as_feature": True, "reason": "Continuous geographic feature, may capture regional patterns beyond state boundary"},
    {"column": "longitude", "role": "Geography", "used_as_feature": True, "reason": "Same as latitude"},
])
print(feature_roles.to_string(index=False))


                       column                 role used_as_feature                                                                                                                                                                             reason
                  provider_id           Identifier           False                                                                                           Unique ID - explicitly excluded per rule: do not use provider ID as a predictive feature
           acceptsassignement         Target (raw)           False                                                                                                                       Original target string - use acceptsassignement_bool instead
      acceptsassignement_bool               Target           False                                                                                                                                                     This is what we are predicting
               A

In [3]:
# Check the relationship between date_reliability_flag and target more rigorously
flag_vs_target = pd.crosstab(df["date_reliability_flag"], df["acceptsassignement_bool"], normalize="index") * 100
print(flag_vs_target.round(2))
print()
# 8.8 percentage point gap exists (52.27% vs 43.44%) per EDA - moderate signal, not leakage since
# the flag is derived from participationbegindate which is known BEFORE acceptance behavior is observed,
# not derived from the target itself. Safe to include as a feature.
print("Decision: date_reliability_flag is a legitimate feature (not leakage) since it's derived from")
print("an independent field (participationbegindate), not from the target. Will be included as a binary indicator.")

acceptsassignement_bool     False  True 
date_reliability_flag                   
Likely genuine date         47.73  52.27
Suspected placeholder date  56.56  43.44

Decision: date_reliability_flag is a legitimate feature (not leakage) since it's derived from
an independent field (participationbegindate), not from the target. Will be included as a binary indicator.


In [4]:
# Use a fixed reference date (today, or dataset's max date) so tenure is a stable numeric feature
reference_date = pd.to_datetime(df["participationbegindate_parsed"]).max()
df["tenure_days"] = (reference_date - pd.to_datetime(df["participationbegindate_parsed"])).dt.days

df["is_suspected_placeholder_date"] = (df["date_reliability_flag"] == "Suspected placeholder date").astype(int)

print("tenure_days stats:")
print(df["tenure_days"].describe())
print()
print("is_suspected_placeholder_date distribution:")
print(df["is_suspected_placeholder_date"].value_counts())

tenure_days stats:
count    57197.000000
mean      4592.754795
std       3704.558274
min          0.000000
25%       1583.000000
50%       3493.000000
75%       7415.000000
max      31592.000000
Name: tenure_days, dtype: float64

is_suspected_placeholder_date distribution:
is_suspected_placeholder_date
0    51728
1     5469
Name: count, dtype: int64


In [5]:
# Multi-hot encoding: one binary column per distinct specialty value
specialty_dummies = df["specialitieslist"].str.get_dummies(sep="|").add_prefix("specialty_")
supply_dummies = df["supplieslist"].str.get_dummies(sep="|").add_prefix("supply_")

print("Number of distinct specialty columns created:", specialty_dummies.shape[1])
print("Number of distinct supply columns created:", supply_dummies.shape[1])
print()
print("Sample specialty columns:", specialty_dummies.columns.tolist()[:10])


Number of distinct specialty columns created: 20
Number of distinct supply columns created: 87

Sample specialty columns: ['specialty_Certified Other', 'specialty_Department Store', 'specialty_Grocery Store', 'specialty_Home Health Agency', 'specialty_Hospital', 'specialty_MSC With Orthotic Personnel', 'specialty_MSC With Orthotic-Prosthetic', 'specialty_MSC With Prosthetic Personnel', 'specialty_MSC With Respiratory Therapist', 'specialty_Medical Supply Company Other']


In [7]:
feature_cols_numeric = ["latitude", "longitude", "tenure_days", "is_suspected_placeholder_date"]

X = pd.concat([
    df[feature_cols_numeric],
    state_dummies,
    specialty_dummies,
    supply_dummies
], axis=1)

y = df["acceptsassignement_bool"].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print()
print("Target class balance:")
print(y.value_counts(normalize=True).round(4) * 100)

NameError: name 'state_dummies' is not defined

In [8]:
state_dummies = pd.get_dummies(df["practicestate"], prefix="state")
print("Number of state columns created:", state_dummies.shape[1])


Number of state columns created: 55


In [9]:
feature_cols_numeric = ["latitude", "longitude", "tenure_days", "is_suspected_placeholder_date"]

X = pd.concat([
    df[feature_cols_numeric],
    state_dummies,
    specialty_dummies,
    supply_dummies
], axis=1)

y = df["acceptsassignement_bool"].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print()
print("Target class balance:")
print(y.value_counts(normalize=True).round(4) * 100)


Feature matrix shape: (57197, 166)
Target shape: (57197,)

Target class balance:
acceptsassignement_bool
1    51.43
0    48.57
Name: proportion, dtype: float64


In [11]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42  # fixed seed for reproducibility, per rule #16

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y  # preserve class balance in both splits, since target is used for stratification
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print()
print("Train target balance:")
print(y_train.value_counts(normalize=True).round(4) * 100)
print()
print("Test target balance:")
print(y_test.value_counts(normalize=True).round(4) * 100)

Train shape: (45757, 166)
Test shape: (11440, 166)

Train target balance:
acceptsassignement_bool
1    51.43
0    48.57
Name: proportion, dtype: float64

Test target balance:
acceptsassignement_bool
1    51.43
0    48.57
Name: proportion, dtype: float64


In [12]:
import os
os.makedirs("../data/interim", exist_ok=True)

# Save full feature matrix + target together for reference
X_full_export = X.copy()
X_full_export["target"] = y
X_full_export["provider_id"] = df["provider_id"]  # keep for traceability only, NOT as a model input
X_full_export.to_csv("../data/interim/model_features_full.csv", index=False)

# Save train/test splits as separate files so 05_model_training.ipynb uses the exact same split
X_train.to_csv("../data/interim/X_train.csv", index=False)
X_test.to_csv("../data/interim/X_test.csv", index=False)
y_train.to_csv("../data/interim/y_train.csv", index=False)
y_test.to_csv("../data/interim/y_test.csv", index=False)

print("Exported:")
print("- data/interim/model_features_full.csv")
print("- data/interim/X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print()
print(f"Random seed used: {RANDOM_SEED} (fixed for reproducibility)")

Exported:
- data/interim/model_features_full.csv
- data/interim/X_train.csv, X_test.csv, y_train.csv, y_test.csv

Random seed used: 42 (fixed for reproducibility)
